# GC Screening (client-mode PySpark)

Pure-Python port of `scripts/spark_submit/run-screening.sh`. The driver runs in this
notebook kernel (client mode); only the executors run in the Spark-on-K8s cluster.
The SQL to run is written **directly as a string** in the cell below (no `qN.sql`
files are loaded anymore).

Open this notebook from the `scripts/pyspark/` directory. Every setting is tuned in
the cells below; the logic lives in `screening.py` / `tpc_pyspark.py`. Executor sizing
is fixed when the session starts, so changing a config requires a fresh session
(build → run → stop).

In [25]:
import os, sys, datetime

# Make screening.py / tpc_pyspark.py importable regardless of the kernel CWD.
SCRIPT_DIR = os.path.abspath(os.environ.get("PYSPARK_SCRIPT_DIR", os.getcwd()))
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# --- AQE log capture -> dedicated file (analyze offline) ---------------------
# Client mode: the driver JVM (this kernel) is launched at the first build_spark
# / getOrCreate, so log4j must be pointed at our config via PYSPARK_SUBMIT_ARGS
# BEFORE that. Re-run this cell only matters before the first session is built;
# once the JVM is up the appender is fixed for the kernel's lifetime.
_ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
AQE_LOG_FILE = os.environ.get("AQE_LOG_FILE", f"/var/spark-logs/aqe-logs/aqe-{_ts}.log")
_log4j_cfg = os.path.join(SCRIPT_DIR, "log4j2-aqe.properties")
_driver_opts = f"-Dlog4j.configurationFile={_log4j_cfg} -Daqe.log.file={AQE_LOG_FILE}"
_base = os.environ.get("PYSPARK_SUBMIT_ARGS", "pyspark-shell")
os.environ["PYSPARK_SUBMIT_ARGS"] = f'--driver-java-options "{_driver_opts}" {_base}'
print("AQE log ->", AQE_LOG_FILE)

import screening
from screening import ScreeningConfig

print("named configs:", sorted(screening.CONFIGS))

AQE log -> /var/spark-logs/aqe-logs/aqe-20260628-193922.log
named configs: ['A', 'B', 'BHJ', 'BHJ2', 'SMJ']


## 1. SQL to run + experiment parameters

Two ways to supply the query:
- Set `SQL_FILE` to a bundled query, e.g. `"tpcds/q9"` or a full path like
  `"/work/queries/tpch/q1.sql"` (TPC-DS/TPC-H `.sql` files are baked at `/work/queries/`), or
- Leave `SQL_FILE = None` and write the query inline in `SQL`.

`SQL_FILE` wins when set; the effective query is `QUERY`. When `REGISTER=True`, the TPC-DS /
TPC-H Parquet tables are registered as temp views first, so the SQL can reference them
directly (e.g. `FROM store_sales ...`).

In [26]:
# Either point SQL_FILE at a bundled query, or leave it None and write SQL inline below.
# e.g. "tpcds/q9" or "/work/queries/tpch/q1.sql"; None => use SQL string
SQL_FILE = "tpch/q9"

SQL = """
select
	nation,
	o_year,
	sum(amount) as sum_profit
from
	(
		select
			n_name as nation,
			year(o_orderdate) as o_year,
			l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity as amount
		from
			part,
			supplier,
			lineitem,
			partsupp,
			orders,
			nation
		where
			s_suppkey = l_suppkey
			and ps_suppkey = l_suppkey
			and ps_partkey = l_partkey
			and p_partkey = l_partkey
			and o_orderkey = l_orderkey
			and s_nationkey = n_nationkey
			and p_name like '%green%'
	) as profit
group by
	nation,
	o_year
order by
	nation,
	o_year desc
"""

# Bundled TPC-DS/TPC-H .sql files live at /work/queries/{tpcds,tpch}/. SQL_FILE wins if set.
QUERY = screening.load_sql(SQL_FILE) if SQL_FILE else SQL
print("The SQL:")
print(QUERY)
print()

LABEL = "tpch_q9"      # name used in output/logs (app id, GC log filename)
CONFIG = "A"           # A | B | BHJ | SMJ | BHJ2  (or use CUSTOM_CONFIG below)
GC = "G1"              # G1 | ZGC | ZGCGEN | SHENANDOAH  (executor collector)
BENCHMARK = "tpch"     # tpcds | tpch  (for table registration)
SCALE = 200
DATA_BASE = "file:///mnt/bench"   # TPC-H SF200 staged on hostPath — no S3 reads
JOB_NAME="new-tpch-q9"

REGISTER = True        # auto-register TPC tables as temp views
SHOW_PLANS = True      # print logical / post-AQE physical plans
SHOW_ROWS = 0     

CUSTOM_CONFIG = ScreeningConfig(
    executor_heap="2g", executor_cores=2, executor_instances=2,
    driver_memory="4g", memory_overhead="768m",
    threshold="128MB", static_threshold="-1", name=JOB_NAME,
)

MASTER = None                                  # e.g. "k8s://https://10.0.0.1:6443"
IMAGE = "gihong96/spark-screening:v1"          # Spark 4.1.2 / Java 21; driver == executor image
NAMESPACE = "spark"
SERVICE_ACCOUNT = "spark"
NODE_SELECTOR_SPARK_DATA = "true"              # None disables the node selector

OBJ_STORAGE_ENDPOINT = "https://hel1.your-objectstorage.com"
S3_SECRET_NAME = "s3-creds"                     # k8s secret with executor AWS keys
# Driver S3A credentials (client mode reads s3a:// from the local driver).
# Paste the keys here (or leave None to use the AWS_* env vars / default chain).
AWS_ACCESS_KEY = None   # not needed: driver reads hostPath; event log uses pod AWS_* env
AWS_SECRET_KEY = None

BENCH_HOST_PATH = "/mnt/bench"                  # executor hostPath volume. None = unmounted

# Client-mode driver networking (this notebook IS the driver, running in-cluster).
DRIVER_HOST = os.environ.get("POD_IP")   # pod IP, reachable by executors
DRIVER_BIND_ADDRESS = "0.0.0.0"
DRIVER_PORT = 7078
BLOCK_MANAGER_PORT = 7079

EVENT_LOG_ENABLED = True
EVENT_LOGS_DIR = "s3a://spark-obj-storage/event-logs"

GC_LOGGING = True
SPARK_LOGS_BASE_DIR = "/var/spark-logs"         # PVC mount path
GC_LOGS_DIR = None                              # None => <SPARK_LOGS_BASE_DIR>/gc-logs-raw
SPARK_LOGS_PVC_CLAIM = "spark-logs-pvc"
GC_FILECOUNT = 10
GC_FILESIZE = "20m"

# Any Spark conf not exposed above goes here (overrides everything else).
EXTRA_CONF = {
    # Driver Web UI (:4040) is reachable through jupyter-server-proxy at
    # http://<host>/jupyter/proxy/4040/ ; proxyBase fixes UI asset/link paths
    # under that prefix. Pin the port so the proxy path is stable (note: a 2nd
    # concurrent session would fall back to 4041).
    "spark.ui.port": "4040",
    "spark.ui.proxyBase": "/jupyter/proxy/4040",
    # "spark.sql.shuffle.partitions": "64",

    # --- AQE logging (captured to AQE_LOG_FILE via log4j2-aqe.properties) ---
    # "Plan changed" / "Final plan" / "Re-optimize" already fire at DEBUG
    # (spark.sql.adaptive.logLevel default). These add the per-rule/batch
    # plan diffs (AQE Query Stage Optimization, AQE Replanning, ...).
    "spark.sql.adaptive.logLevel": "DEBUG",
    "spark.sql.planChangeLog.enabled": "true",
    "spark.sql.planChangeLog.level": "DEBUG",
}

The SQL:
-- using default substitutions

select
	nation,
	o_year,
	sum(amount) as sum_profit
from
	(
		select
			n_name as nation,
			year(o_orderdate) as o_year,
			l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity as amount
		from
			part,
			supplier,
			lineitem,
			partsupp,
			orders,
			nation
		where
			s_suppkey = l_suppkey
			and ps_suppkey = l_suppkey
			and ps_partkey = l_partkey
			and p_partkey = l_partkey
			and o_orderkey = l_orderkey
			and s_nationkey = n_nationkey
			and p_name like '%green%'
	) as profit
group by
	nation,
	o_year
order by
	nation,
	o_year desc




## 5. Build session → run → stop

In [27]:
CFG = CUSTOM_CONFIG if CUSTOM_CONFIG is not None else CONFIG

spark = screening.build_spark(
    CFG,
    label=LABEL,
    benchmark=BENCHMARK,
    scale=SCALE,
    gc=GC,
    master=MASTER,
    image=IMAGE,
    namespace=NAMESPACE,
    service_account=SERVICE_ACCOUNT,
    node_selector_spark_data=NODE_SELECTOR_SPARK_DATA,
    obj_storage_endpoint=OBJ_STORAGE_ENDPOINT,
    s3_secret_name=S3_SECRET_NAME,
    aws_access_key=AWS_ACCESS_KEY,
    aws_secret_key=AWS_SECRET_KEY,
    bench_host_path=BENCH_HOST_PATH,
    driver_host=DRIVER_HOST,
    driver_bind_address=DRIVER_BIND_ADDRESS,
    driver_port=DRIVER_PORT,
    block_manager_port=BLOCK_MANAGER_PORT,
    event_log_enabled=EVENT_LOG_ENABLED,
    event_logs_dir=EVENT_LOGS_DIR,
    gc_logging=GC_LOGGING,
    spark_logs_base_dir=SPARK_LOGS_BASE_DIR,
    gc_logs_dir=GC_LOGS_DIR,
    spark_logs_pvc_claim=SPARK_LOGS_PVC_CLAIM,
    gc_filecount=GC_FILECOUNT,
    gc_filesize=GC_FILESIZE,
    extra_conf=EXTRA_CONF,
)
spark

In [ ]:
result = screening.run_screening(
    spark,
    QUERY,
    CFG,
    label=LABEL,
    benchmark=BENCHMARK,
    scale=SCALE,
    data_base=DATA_BASE,
    register=REGISTER,
    show_plans=SHOW_PLANS,
    show_rows=SHOW_ROWS,
)
result

In [24]:
# Free the cluster resources before building a session for another config.
spark.stop()

## (optional) Sweep the same SQL across several GC collectors

Build a fresh session per collector (G1 / ZGC / generational ZGC / Shenandoah) for the
chosen `CONFIG` and collect the `RESULT` dicts for later analysis. Each run's executor GC
log lands on the PVC tagged with the collector name.

In [ ]:
results = []
for gc_name in ["G1", "ZGC", "ZGCGEN", "SHENANDOAH"]:
    s = screening.build_spark(CFG, label=LABEL, benchmark=BENCHMARK, scale=SCALE,
                              gc=gc_name, master=MASTER, image=IMAGE, namespace=NAMESPACE)
    try:
        results.append(
            screening.run_screening(s, QUERY, CFG, label=LABEL, benchmark=BENCHMARK,
                                    scale=SCALE, data_base=DATA_BASE, register=REGISTER)
        )
    finally:
        s.stop()

results